## Define Path(s) of Rater 1 + 2

In [4]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

# Put paths here
r1_files = ['/Users/chris/Desktop/IRR CSVs (all 3 agents)/DUI/RAs/Kyle/Classify_Element_DUI_Kyle - Classifcations.csv',
'/Users/chris/Desktop/IRR CSVs (all 3 agents)/Hate Crimes/RAs/Kyle/hate_crimes_kyle - Classifcations.csv',
'/Users/chris/Desktop/IRR CSVs (all 3 agents)/Receiving Stolen Propery/RAs/Kyle/Classify_Element_Recieved_stolen_Kyle - Classifcations.csv',
'/Users/chris/Desktop/IRR CSVs (all 3 agents)/Stalking/RAs/Kyle/Stalking_Kyle - Classifcations.csv',
'/Users/chris/Desktop/IRR CSVs (all 3 agents)/Statutory Rape/RAs/Kyle/Copy of Classify_Element - Classifcations.csv',
'/Users/chris/Desktop/IRR CSVs (all 3 agents)/Trespassing/RAs/Kyle/Classify_Element_Trespassing - Classifcations.csv'
]

r2_files = ['/Users/chris/Desktop/IRR CSVs (all 3 agents)/DUI/RAs/Sophia/Classify_Element_DUI_Sophia - Classifcations.csv',
'/Users/chris/Desktop/IRR CSVs (all 3 agents)/Hate Crimes/RAs/Sophia/hate_crimes_Sophia - Classifcations.csv',
'/Users/chris/Desktop/IRR CSVs (all 3 agents)/Receiving Stolen Propery/RAs/Sophia/Classify_Element_Recieved_stolen_Sophia - Classifcations.csv',
'/Users/chris/Desktop/IRR CSVs (all 3 agents)/Stalking/RAs/Sophia/Stalking_Sophia - Classifcations.csv',
'/Users/chris/Desktop/IRR CSVs (all 3 agents)/Statutory Rape/RAs/Sophia/Classify_Stautory_Rape_Sophia - Classifcations.csv',
'/Users/chris/Desktop/IRR CSVs (all 3 agents)/Trespassing/RAs/Sophia/Classify_Element_Trespassing_Sophia - Classifcations.csv'
]

# ---- Load + stack each rater’s CSVs ----
r1 = pd.concat([pd.read_csv(f) for f in r1_files], ignore_index=True)
r2 = pd.concat([pd.read_csv(f) for f in r2_files], ignore_index=True)


## Export Stacked CSVs (optional)

In [3]:
import os

out_dir = "/Users/chris/Desktop/Stacked_class_CSVs"
os.makedirs(out_dir, exist_ok=True)   # creates the folder if it's missing; does nothing if it already exists

r1_out = os.path.join(out_dir, "Rater1_STACKED.csv")
r2_out = os.path.join(out_dir, "Rater2_STACKED.csv")
r1.to_csv(r1_out, index=False)
r2.to_csv(r2_out, index=False)

## Run IRR Metrics

In [5]:

# Define ordinal scale
ORDER = [
    "Strict Liability",
    "Negligent",
    "Reckless",
    "Knowing",
    "Purposive",
]
rank_map = {label: i + 1 for i, label in enumerate(ORDER)}  # asigns numbers to each lable in ordinal scale

def irr_for_min_rank_no_id(r1, r2, cols, prefix, rank_map):
    r1 = r1.copy()
    r2 = r2.copy()

    def clean_to_na(s):
        # normalize to string, strip, then map blanks + "No Mental States" to NA
        s = s.astype("string").str.strip()
        s = s.replace("", pd.NA)
        s = s.replace("No Mental States", pd.NA)
        return s

    # Clean + map to ranks
    for col in cols:
        r1[col] = clean_to_na(r1[col])
        r2[col] = clean_to_na(r2[col])

        r1[f"{prefix}_{col}_rank"] = r1[col].map(rank_map)
        r2[f"{prefix}_{col}_rank"] = r2[col].map(rank_map)

    rank_cols_r1 = [f"{prefix}_{col}_rank" for col in cols]
    rank_cols_r2 = [f"{prefix}_{col}_rank" for col in cols]

    # Min rank across the three columns (skipna=True means:
    # - if at least one rank exists, min is computed
    # - if all are NA, result is NA)
    r1[f"{prefix}_min_rank"] = r1[rank_cols_r1].min(axis=1, skipna=True)
    r2[f"{prefix}_min_rank"] = r2[rank_cols_r2].min(axis=1, skipna=True)

    # Align by row order (index)
    merged = pd.DataFrame({
        f"{prefix}_min_rank_r1": r1[f"{prefix}_min_rank"].reset_index(drop=True),
        f"{prefix}_min_rank_r2": r2[f"{prefix}_min_rank"].reset_index(drop=True),
    })

    # --- (A) NA vs non-NA agreement metrics (binary) ---
    merged[f"{prefix}_is_na_r1"] = merged[f"{prefix}_min_rank_r1"].isna()
    merged[f"{prefix}_is_na_r2"] = merged[f"{prefix}_min_rank_r2"].isna()

    na_agreement = (merged[f"{prefix}_is_na_r1"] == merged[f"{prefix}_is_na_r2"]).mean() * 100

    # Kappa for NA vs non-NA, but only if both classes appear
    if merged[f"{prefix}_is_na_r1"].nunique() > 1 and merged[f"{prefix}_is_na_r2"].nunique() > 1:
        kappa_na = cohen_kappa_score(merged[f"{prefix}_is_na_r1"], merged[f"{prefix}_is_na_r2"])
    else:
        kappa_na = float("nan")

    # --- (B) Ordinal kappa on ranks, excluding NA rows ---
    merged_ok = merged.dropna(subset=[f"{prefix}_min_rank_r1", f"{prefix}_min_rank_r2"]).copy()

    pa = (merged_ok[f"{prefix}_min_rank_r1"] == merged_ok[f"{prefix}_min_rank_r2"]).mean() * 100 if len(merged_ok) else float("nan")

    if len(merged_ok) > 0:
        kappa_linear = cohen_kappa_score(
            merged_ok[f"{prefix}_min_rank_r1"],
            merged_ok[f"{prefix}_min_rank_r2"],
            weights="linear",
        )
        kappa_quadratic = cohen_kappa_score(
            merged_ok[f"{prefix}_min_rank_r1"],
            merged_ok[f"{prefix}_min_rank_r2"],
            weights="quadratic",
        )
    else:
        kappa_linear = float("nan")
        kappa_quadratic = float("nan")

    print(f"({prefix}_min): Aligned rows  {len(merged)} total")
    print(f"({prefix}_min): Used for ordinal kappa  {len(merged_ok)} (both non-missing)")
    print(f"({prefix}_min): Percent agreement on ordinal ranks {pa:.1f}%")
    print(f"({prefix}_min): Weighted kappa (linear) {kappa_linear:.3f}")
    print(f"({prefix}_min): Weighted kappa (quadratic)  {kappa_quadratic:.3f}")
    print(f"({prefix}_min): NA agreement {na_agreement:.1f}%")
    print(f"({prefix}_min): Kappa for NA vs non-NA  {kappa_na:.3f}")

    return merged, merged_ok

merged_action, merged_action_ok = irr_for_min_rank_no_id(
    r1, r2,
    cols=["Action #1", "Action #2", "Action #3"],
    prefix="Action",
    rank_map=rank_map,
)

merged_result, merged_result_ok = irr_for_min_rank_no_id(
    r1, r2,
    cols=["Result #1", "Result #2", "Result #3"],
    prefix="Result",
    rank_map=rank_map,
)

merged_circumstance, merged_circumstance_ok = irr_for_min_rank_no_id(
    r1, r2,
    cols=["Circumstance #1", "Circumstance #2", "Circumstance #3"],
    prefix="Circumstance",
    rank_map=rank_map,
)


(Action_min): Aligned rows  313 total
(Action_min): Used for ordinal kappa  89 (both non-missing)
(Action_min): Percent agreement on ordinal ranks 70.8%
(Action_min): Weighted kappa (linear) 0.373
(Action_min): Weighted kappa (quadratic)  0.216
(Action_min): NA agreement 74.1%
(Action_min): Kappa for NA vs non-NA  0.473
(Result_min): Aligned rows  313 total
(Result_min): Used for ordinal kappa  10 (both non-missing)
(Result_min): Percent agreement on ordinal ranks 100.0%
(Result_min): Weighted kappa (linear) 1.000
(Result_min): Weighted kappa (quadratic)  1.000
(Result_min): NA agreement 89.1%
(Result_min): Kappa for NA vs non-NA  0.322
(Circumstance_min): Aligned rows  313 total
(Circumstance_min): Used for ordinal kappa  74 (both non-missing)
(Circumstance_min): Percent agreement on ordinal ranks 55.4%
(Circumstance_min): Weighted kappa (linear) 0.279
(Circumstance_min): Weighted kappa (quadratic)  0.336
(Circumstance_min): NA agreement 76.0%
(Circumstance_min): Kappa for NA vs non-N